# Notebook 07: EM-side classifier diagnostics

This notebook presents diagnostic outputs from the EM-side
classifier and validates that the spectral features
discriminate anomalous executions on a held-out test split.

As before, the demonstrator is independent of the paper's
headline numbers: the goal is to expose the classifier's
behaviour, not to match a specific reported precision or
recall.


## Reference figure: feature importance

Per-feature importance ranking from the EM analysis pipeline
(reference figure). The bar plot shows the relative weight that
the production classifier assigns to each of the 21 spectral
features when separating normal from anomalous traces. The
ordering on this capture is not directly comparable to the
baseline ranking computed below, because the production
classifier and the baseline use different model configurations
and feature scaling.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import IFrame, display
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

FEATURES_CSV = Path("..") / "data" / "picoc_features.csv"
FIGURES_DIR = Path("..") / "figures" / "em_evidence"

display(IFrame(str(FIGURES_DIR / "em_feature_importance.pdf"), width=800, height=600))


## Reference figure: detection metrics

Precision, recall, and F1 scores reported by the EM analysis
pipeline on this capture. The numbers are tied to the specific
classifier configuration used in the production pipeline and
are not directly comparable to the baseline reproduced below.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_detection_metrics.pdf"), width=800, height=600))


## Reference figure: anomaly distribution

Distribution of anomalous-trace scores along the classifier's
decision axis, with the chosen operating threshold marked.


In [ ]:
display(IFrame(str(FIGURES_DIR / "em_anomaly_distribution.pdf"), width=800, height=600))


## Baseline reproduction: random forest on 21 features

This is a baseline reproduction. The paper's pipeline uses a
different classifier configuration; here a random forest with
default hyper-parameters is fit on an 80/20 stratified split
and the resulting classification report and confusion matrix
are printed.


In [ ]:
df = pd.read_csv(FEATURES_CSV)
feature_cols = ['rms', 'mean', 'std_dev', 'crest_factor', 'peak_to_peak',
                'entropy', 'peak_count', 'kurtosis', 'skewness', 'zcr',
                'energy_low', 'energy_mid', 'energy_high',
                'peak_freq', 'peak_magnitude', 'spectral_entropy',
                'spectral_crest', 'harmonic_distortion',
                'spectral_flatness', 'spectral_rolloff', 'hnr']
X = df[feature_cols].fillna(0).values
y = df["is_anomalous"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Classification report (held-out test, 20%):")
print(classification_report(y_test, y_pred, target_names=["normal", "anomalous"]))
print("Confusion matrix (rows = true, cols = predicted):")
print(confusion_matrix(y_test, y_pred))


## Top-ranked features in the baseline


In [ ]:
importances = pd.Series(clf.feature_importances_, index=feature_cols)
ranked = importances.sort_values(ascending=False)
print("Top 10 features by Gini importance (baseline RF):")
print(ranked.head(10).to_string())

fig, ax = plt.subplots(figsize=(7, 5))
top = ranked.head(10).iloc[::-1]
ax.barh(top.index, top.values, color="#1f4e79")
ax.set_xlabel("Gini importance")
ax.set_title("Baseline RF: top 10 features")
ax.grid(True, axis="x", linestyle=":", alpha=0.5)
fig.tight_layout()
plt.show()


## Closing

For the design rationale of the 21-feature space and the
calibration setup that defines normal vs. anomalous, see
[`docs/EM_METHODOLOGY.md`](../docs/EM_METHODOLOGY.md). For the
paper's headline numbers (R_certain, p_excl, R_bug ratios,
MTTFB), see notebooks 01-04 and the CSVs under `data/`.
